In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import csv
import os

if not os.path.exists("test_notebooks"):
    os.chdir("..")

assert os.path.exists("test_notebooks")

In [23]:
# benchmark dataset
# 目标: 12个tag进行rag，3秒内
from entropy.domain.services.tag_checker import TagChecker


input_data = """
shimmering, pleading, soaked, slightly open mouth, thin straps, sheer dress, blushing, shoulderless dress,
seifuku, japanese school uniform, bubbles, knitwear
"""

input_data = TagChecker.extract_all_tags(input_data)
assert len(input_data) == 12

print(",".join(input_data))

shimmering,pleading,soaked,slightly open mouth,thin straps,sheer dress,blushing,shoulderless dress,seifuku,japanese school uniform,bubbles,knitwear


In [24]:
from entropy.domain.services.rag_service import RagService

In [25]:
# 非batch

result1 = []

for invalid_tag in input_data:
    tags, scores = RagService.do_rag(query_text=invalid_tag, rerank_output=10)
    result1.append(tags)

Compute Scores: 100%|██████████| 4/4 [00:00<00:00,  6.99it/s]


In [26]:
# batch
batch_rag_output = RagService.batch_rag(query_text_list=input_data, rerank_output=10)

result2 = [tags for tags, scores in batch_rag_output]

Compute Scores: 100%|██████████| 47/47 [00:05<00:00,  9.00it/s]


In [27]:
import json


for (r1, r2) in zip(result1, result2):
    if not json.dumps(r1) == json.dumps(r2):
        print("r1:", r1)
        print("r2:", r2)

r1: ['layered dress', 'covering body', 'layered clothes', 'unbuttoned dress', 'revealing clothes', 'sweater dress', 'shiny clothes', 'tight dress', 'wringing dress', 'patterned clothing']
r2: ['layered dress', 'covering body', 'layered clothes', 'unbuttoned dress', 'revealing clothes', 'sweater dress', 'shiny clothes', 'wringing dress', 'tight dress', 'patterned clothing']
r1: ['bubble', 'bugles', 'bobbles', 'in bubble', 'blob', 'bauble', 'bulge', 'bulges touching', 'bule', 'bangle']
r2: ['bubble', 'bugles', 'bobbles', 'in bubble', 'blob', 'bauble', 'bulge', 'bulges touching', 'bangle', 'bule']
